---
title: Pemantauan fase tanam dan kelompok jadwal panen padi dengan Sentinel-2
short_title: Pemantauan Fase Padi
subject: Studi Kasus
subtitle: Mengidentifikasi lahan sawah irigasi, memadukan dinamika curah hujan, dan memetakan pergeseran jadwal tanam antarblok di Karawang.
description: Menggunakan citra Sentinel-2 L2A di Open Data Cube dan curah hujan harian ERA5 (Open-Meteo) untuk mendeteksi sawah berbasis sinyal penggenangan dan vegetasi (LSWI, NDVI, EVI), memetakan komposisi fase bulanan, serta memetakan kelompok jadwal tanam awal, utama, dan susulan di sekitar Kecamatan Telagasari, Kabupaten Karawang.
authors:
  - name: Muhammad Taufik
    github: taufik-shf
    roles:
      - writing
      - methodology
      - formal analysis
      - software
      - visualization
      - data curation
  - name: Gatot Pramono
    roles:
      - conceptualization
keywords:
  - sentinel-2
  - padi
  - fenologi
  - karawang
  - ndvi
  - lswi
  - evi
  - irigasi
---

## A. Latar belakang

Di wilayah sawah irigasi seperti pantai utara Jawa Barat, petak-petak sawah jarang ditanam dan dipanen pada waktu yang sama karena mengikuti giliran air dan kesiapan lahan. Akibatnya, satu hamparan sawah bisa berada pada fase pertumbuhan yang berbeda-beda dalam waktu yang berdekatan. Kondisi ini membuat pengaturan beberapa kegiatan di lapangan, seperti giliran air, kesiapan alat, maupun waktu panen, menjadi lebih menantang karena kebutuhannya tidak seragam di seluruh area.

Citra satelit multiwaktu dapat membantu memantau keragaman tersebut secara spasial dari petak ke petak. Sepanjang satu musim tanam, sawah irigasi menunjukkan perubahan permukaan yang jelas: diawali genangan air saat olah tanah dan awal tanam, berangsur menghijau hingga puncak tutupan daun, lalu mengering menjelang panen [(Xiao dkk., 2005)](#ref-xiao-2005). Perubahan antara air permukaan dan kehijauan tanaman itu terekam melalui indeks kelembapan (*Land Surface Water Index*, **LSWI**) bersama indeks vegetasi (**NDVI** dan **EVI**) [(Rouse dkk., 1974)](#ref-rouse-1974); [(Huete dkk., 2002)](#ref-huete-2002); [(Xiao dkk., 2006)](#ref-xiao-2006).

Studi kasus ini menerapkan pendekatan tersebut pada citra Sentinel-2 Level-2A (`s2_l2a`) di sekitar Kecamatan Telagasari, Kabupaten Karawang, selama musim tanam kedua (Mei hingga November 2025) yang relatif cerah. Selain melacak pola pertumbuhannya, data satelit juga disandingkan dengan curah hujan harian global [(Hersbach dkk., 2020)](#ref-hersbach-2020) untuk melihat sejauh mana pertumbuhan padi pada periode kering ini ditopang pasokan irigasi. Dari seri waktu yang terbentuk, area sawah selanjutnya dipetakan menjadi kelompok blok tanam awal, utama, dan akhir beserta komposisi fase bulanannya sebagai alat bantu pembanding kondisi di lapangan.

## B. Menyiapkan data dan wilayah kajian

In [ ]:
import datacube
import folium
import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr
import pandas as pd
import requests
from ipyleaflet import basemaps
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from shapely.geometry import box

Wilayah kajian (*Area of Interest*, AoI) mencakup kawasan persawahan di Kabupaten Karawang.

In [ ]:
min_lon, max_lon = 107.390, 107.440
min_lat, max_lat = -6.315, -6.290
bbox = (min_lon, min_lat, max_lon, max_lat)

aoi = gpd.GeoDataFrame(geometry=[box(*bbox)], crs="EPSG:4326")
aoi.explore(tiles=basemaps.Esri.WorldImagery)

Data `s2_l2a` dimuat pada sistem koordinat UTM zona 48S (`EPSG:32748`, mencakup Jawa Barat) dengan resolusi 10 meter. Agar tidak mengunduh seluruh kanal pada tanggal yang tertutup awan tebal, pemuatan dilakukan dua tahap: memeriksa proporsi piksel bersih dari kanal *Scene Classification Layer* (`scl`) lebih dulu, kemudian memuat kanal optis hanya pada tanggal yang memiliki cakupan bersih minimal 75%.

In [ ]:
dc = datacube.Datacube(app="pemantauan_pertumbuhan_padi")
time_range = ("2025-05-01", "2025-11-15")
crs = "EPSG:32748"
resolution = (-10, 10)

scl_ds = dc.load(
    product="s2_l2a",
    x=(min_lon, max_lon),
    y=(min_lat, max_lat),
    time=time_range,
    measurements=["scl"],
    resolution=resolution,
    group_by="solar_day",
    output_crs=crs,
    dask_chunks={"time": 1, "x": 512, "y": 512},
)
scl_ds["scl"]

In [ ]:
# Tandai piksel bersih (vegetasi=4, tanah terbuka=5, air=6, unclassified=7)
clear_init = scl_ds["scl"].isin([4, 5, 6, 7])

# Hitung persentase piksel bersih per tanggal akuisisi
clear_pct = (100.0 * clear_init.mean(dim=("y", "x"))).compute()

# Ambil tanggal yang memenuhi ambang cakupan bersih minimal 75%
good_time = clear_pct.time.where(clear_pct >= 75.0, drop=True)

# Ubah ke set string 'YYYY-MM-DD' untuk pencocokan tanggal akuisisi
good_days = set(pd.to_datetime(good_time.values).strftime("%Y-%m-%d"))

print(f"Cakupan bersih >= 75%: {len(good_days)} dari {scl_ds.sizes['time']} akuisisi")

Daftar tanggal bersih kemudian digunakan untuk menyaring dataset sebelum memuat kanal `blue`, `green`, `red`, `nir`, `swir16`, dan `scl`.

In [ ]:
datasets = dc.find_datasets(
    product="s2_l2a",
    x=(min_lon, max_lon),
    y=(min_lat, max_lat),
    time=time_range,
)
filtered_datasets = [d for d in datasets if str(d.center_time)[:10] in good_days]

ds = dc.load(
    datasets=filtered_datasets,
    x=(min_lon, max_lon),
    y=(min_lat, max_lat),
    measurements=["blue", "green", "red", "nir", "swir16", "scl"],
    resolution=resolution,
    group_by="solar_day",
    output_crs=crs,
    dask_chunks={"time": 1, "x": 512, "y": 512},
).compute()

ds

Piksel awan, bayangan, dan data kosong disaring lebih dulu agar nilai indeks tidak terdistorsi gangguan atmosfer atau nilai nol. Kanal optis kemudian diubah ke skala reflektansi 0–1 bertipe `float32` supaya selisih negatif antarkanal tetap terjaga dan sesuai dengan asumsi skala pada rumus EVI (`L = 1`) [(Huete dkk., 2002)](#ref-huete-2002).

In [ ]:
# Mask piksel bersih (SCL 4-7) dan nilai band valid 
clear = ds["scl"].isin([4, 5, 6, 7]) & (ds["red"] > 0)

# Ubah kanal ke float32 skala reflektansi 0-1
blue = ds["blue"].where(clear).astype("float32") / 10000.0
green = ds["green"].where(clear).astype("float32") / 10000.0
red = ds["red"].where(clear).astype("float32") / 10000.0
nir = ds["nir"].where(clear).astype("float32") / 10000.0
swir1 = ds["swir16"].where(clear).astype("float32") / 10000.0

# Cek kondisi data
red.isel(time=0).values

## C. Menghitung indeks spektral dan memetakan indikasi sawah

Tiga indeks spektral dihitung dari kanal reflektansi yang telah dibersihkan:

- **NDVI** $= \dfrac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$ [(Rouse dkk., 1974)](#ref-rouse-1974)
- **EVI** $= 2{,}5 \times \dfrac{\text{NIR} - \text{Red}}{\text{NIR} + 6\,\text{Red} - 7{,}5\,\text{Blue} + 1}$ [(Huete dkk., 2002)](#ref-huete-2002)
- **LSWI** $= \dfrac{\text{NIR} - \text{SWIR1}}{\text{NIR} + \text{SWIR1}}$ [(Xiao dkk., 2005)](#ref-xiao-2005)

In [ ]:
ndvi = (nir - red) / (nir + red)
evi = 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0)
lswi = (nir - swir1) / (nir + swir1)

Merujuk kriteria [Xiao dkk. (2006)](#ref-xiao-2006), deteksi sawah irigasi dilakukan bertahap. 

Pertama, mencari piksel yang mengalami penggenangan pada periode transisi menuju awal tanam (Mei hingga Juli 2025), yaitu saat kelembapan permukaan (`LSWI + 0,05`) mendekati atau melampaui indeks vegetasi sementara NDVI masih rendah (`< 0,35`).

In [ ]:
# Sinyal genangan awal tanam: LSWI mendekati/lebih dari EVI/NDVI saat vegetasi rendah
early_window = slice("2025-05-01", "2025-07-31")

flood_signal = (
    (lswi.sel(time=early_window) + 0.05 >= evi.sel(time=early_window))
    | (lswi.sel(time=early_window) + 0.05 >= ndvi.sel(time=early_window))
) & (ndvi.sel(time=early_window) < 0.35)

has_flood = flood_signal.any(dim="time")

Agar badan air permanen (seperti saluran irigasi, sungai, atau kolam) tidak ikut terhitung, piksel yang tergenang tadi disaring lagi dengan syarat mengalami pertumbuhan kanopi rapat (`NDVI >= 0,55`) pada paruh musim berikutnya (pertengahan Juli hingga pertengahan Oktober 2025).

In [ ]:
# Syarat puncak vegetasi untuk memisahkan sawah dari genangan non-vegetasi (air permanen)
peak_window = slice("2025-07-15", "2025-10-15")
has_veg_peak = ndvi.sel(time=peak_window).max(dim="time") >= 0.55

paddy_mask = has_flood & has_veg_peak
permanent_water = has_flood & ~has_veg_peak

Tiap piksel beresolusi 10 x 10 meter setara `0,01 ha`, sehingga luas indikasi sawah dan area non-vegetasi yang tersaring dapat langsung dihitung.

In [ ]:
# Hitung luas per piksel (ha) dan total luas per kategori
pixel_area_ha = (abs(resolution[0]) * abs(resolution[1])) / 10_000
total_aoi_ha = paddy_mask.size * pixel_area_ha
paddy_ha = float(paddy_mask.sum()) * pixel_area_ha
excluded_ha = float(permanent_water.sum()) * pixel_area_ha

print(f"Luas AoI                : {total_aoi_ha:,.1f} ha")
print(f"Indikasi lahan sawah    : {paddy_ha:,.1f} ha ({paddy_ha / total_aoi_ha * 100:.1f}%)")
print(f"Tersaring (non-vegetasi): {excluded_ha:,.1f} ha")

Untuk memeriksa kesesuaian hasil klasifikasi secara visual, sebaran `paddy_mask` dibandingkan langsung dengan citra warna alami (*true color* RGB) saat tutupan vegetasi tinggi.

In [ ]:
def plot_map(ax, base, title="", overlays=None, legend=None, cmap="YlGn", vmin=0, vmax=1, cbar_label=None):
    if base.ndim == 3:
        base.plot.imshow(ax=ax, rgb="band", add_labels=False)
    else:
        base.plot.imshow(
            ax=ax,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            add_colorbar=bool(cbar_label),
            add_labels=False,
            cbar_kwargs={"label": cbar_label} if cbar_label else None,
        )
    for mask, mask_cmap, alpha in overlays or []:
        cmap_obj = ListedColormap([mask_cmap]) if isinstance(mask_cmap, str) and mask_cmap.startswith("#") else mask_cmap
        mask.where(mask).plot.imshow(
            ax=ax, cmap=cmap_obj, vmin=0, vmax=1, alpha=alpha, add_colorbar=False, add_labels=False
        )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.tick_params(axis="both", labelsize=8)
    ax.locator_params(axis="x", nbins=4)
    if legend:
        ax.legend(handles=[Patch(facecolor=c, label=t) for c, t in legend], loc="lower left", fontsize=8)

In [ ]:
rgb_peak = (
    ds[["red", "green", "blue"]]
    .sel(time="2025-09-05")
    .isel(time=0)
    .to_array(dim="band")
    .astype("float32")
    / 3000.0
).clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
plot_map(axes[0], rgb_peak, "Citra warna alami Sentinel-2 (2025-09-05)")
plot_map(
    axes[1],
    rgb_peak,
    f"Indikasi sawah ({paddy_ha:,.1f} ha)",
    overlays=[(paddy_mask, "#22c55e", 0.65), (permanent_water, "#0284c7", 0.9)],
    legend=[("#22c55e", "Indikasi sawah"), ("#0284c7", "Air/non-vegetasi tersaring")],
)
plt.tight_layout()
plt.show()

Pada perbandingan tersebut, hamparan sawah hijau di sebagian besar wilayah kajian tertangkap sebagai indikasi sawah, sedangkan area non-sawah seperti permukiman, jaringan jalan, serta badan air permanen tersaring keluar dari `paddy_mask`. Perlu dicatat bahwa klasifikasi spektral ini bersifat indikatif; untuk pelaporan resmi luas tanam atau lahan baku, hasilnya perlu dipadukan dengan poligon Lahan Baku Sawah (LBS).

## D. Melacak kurva fase pertumbuhan

Rata-rata spasial NDVI, EVI, dan LSWI pada piksel `paddy_mask` dihitung untuk tiap tanggal pengamatan bersih agar pola perubahan antarwaktu terlihat jelas.

In [ ]:
# Hitung rata-rata spasial indeks di area sawah untuk tiap tanggal bersih
ts = pd.DataFrame(
    {
        "date": pd.to_datetime(ds.time.values).tz_localize(None),
        "NDVI": ndvi.where(paddy_mask).mean(dim=("y", "x")).values,
        "EVI": evi.where(paddy_mask).mean(dim=("y", "x")).values,
        "LSWI": lswi.where(paddy_mask).mean(dim=("y", "x")).values,
    }
)
ts.round(3).head()

Tabel di atas memperlihatkan peralihan dari sisa tutupan vegetasi awal Mei (`NDVI` sekitar `0,50`) menuju fase olah tanah dan penggenangan pada Juni hingga awal Juli saat NDVI menurun dan LSWI meningkat. Dari seri waktu tersebut, tiga momen kunci diambil dari data: puncak LSWI saat awal tanam (penggenangan), puncak NDVI (puncak vegetatif), dan LSWI terendah pascapuncak sampai akhir Oktober (pemasakan menuju panen).

In [ ]:
# Batasi jendela waktu: awal musim (genangan) dan pascapuncak (pemasakan/panen)
early_ts = ts[ts["date"] <= "2025-07-31"]
after_peak = ts[(ts["date"] >= "2025-09-05") & (ts["date"] <= "2025-10-31")]

# Cari tanggal tiap fase: LSWI tertinggi di awal, NDVI tertinggi, lalu LSWI terendah setelah puncak
t_flood = early_ts.loc[early_ts["LSWI"].idxmax(), "date"]
t_peak = ts.loc[ts["NDVI"].idxmax(), "date"]
t_harvest = after_peak.loc[after_peak["LSWI"].idxmin(), "date"]

# Susun tabel tanggal fase kunci
milestones = pd.DataFrame([
    {"Fase": "Penggenangan / awal tanam", "Tanggal": t_flood.strftime("%Y-%m-%d")},
    {"Fase": "Puncak vegetatif", "Tanggal": t_peak.strftime("%Y-%m-%d")},
    {"Fase": "Pemasakan / panen", "Tanggal": t_harvest.strftime("%Y-%m-%d")},
])

# Gabungkan nilai NDVI, EVI, dan LSWI pada masing-masing tanggal fase
ts_by_day = ts.assign(Tanggal=ts["date"].dt.strftime("%Y-%m-%d"))[
    ["Tanggal", "NDVI", "EVI", "LSWI"]
]
milestones = milestones.merge(ts_by_day, on="Tanggal")
milestones.round(3)

Karena periode Mei hingga November berada di musim kemarau, kurva pertumbuhan perlu disandingkan dengan curah hujan harian untuk melihat sejauh mana penggenangan dan fase tumbuh ditopang oleh hujan lokal atau pasokan irigasi. Data curah hujan diambil dari layanan arsip historis *Open-Meteo* (berbasis reanalisis ERA5) lalu dirangkum per 10 hari (dasarian) [(Hersbach dkk., 2020)](#ref-hersbach-2020).

In [ ]:
# Ambil curah hujan harian ERA5 via Open-Meteo dan agregasikan per 10 hari (dasarian)
rain_resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": float((min_lat + max_lat) / 2),
        "longitude": float((min_lon + max_lon) / 2),
        "start_date": time_range[0],
        "end_date": time_range[1],
        "daily": "precipitation_sum",
        "timezone": "Asia/Jakarta",
    },
    timeout=30,
)
rain_resp.raise_for_status()
rain_daily = pd.DataFrame(
    {
        "date": pd.to_datetime(rain_resp.json()["daily"]["time"]),
        "rain_mm": rain_resp.json()["daily"]["precipitation_sum"],
    }
)
rain_10d = rain_daily.set_index("date").resample("10D")["rain_mm"].sum().reset_index()
rain_10d.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.2))
ax2 = ax.twinx()

# Latar batang curah hujan 10-harian pada sumbu kanan
ax2.bar(
    rain_10d["date"] + pd.Timedelta(days=5),
    rain_10d["rain_mm"],
    width=8,
    color="#93c5fd",
    alpha=0.38,
    label="Curah hujan 10-harian (mm)",
    zorder=1,
)
ax2.set_ylabel("Curah hujan 10-harian (mm)", color="#0369a1")
ax2.set_ylim(0, max(160, rain_10d["rain_mm"].max() * 1.3))
ax2.tick_params(axis="y", colors="#0369a1")

ax.plot(ts["date"], ts["NDVI"], "o-", color="#15803d", linewidth=2, label="NDVI", zorder=3)
ax.plot(ts["date"], ts["EVI"], "s-", color="#65a30d", linewidth=1.8, label="EVI", zorder=3)
ax.plot(ts["date"], ts["LSWI"], "^--", color="#0284c7", linewidth=1.8, label="LSWI", zorder=3)

phase_markers = [
    (t_flood, "Penggenangan /\nawal tanam", "#0284c7"),
    (t_peak, "Puncak\nvegetatif", "#15803d"),
    (t_harvest, "Pemasakan /\npanen", "#b45309"),
]
for dt, label, color in phase_markers:
    ax.axvline(dt, color=color, linestyle=":", linewidth=1.8, zorder=2)
    ax.text(dt, 0.92, f"{label}\n({dt.strftime('%d %b')})", color=color, ha="center", va="top", fontsize=9)

ax.set_title("Kurva indeks spektral sawah dan curah hujan 10-harian (Mei–November 2025)", pad=30)
ax.set_ylabel("Nilai indeks")
ax.set_ylim(0, 0.96)
ax.set_zorder(ax2.get_zorder() + 1)
ax.patch.set_visible(False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.grid(True, alpha=0.3)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=4, frameon=False)
plt.tight_layout()
plt.show()

Kurva memperlihatkan satu siklus tanam penuh selama sekitar 100 hari: setelah puncak penggenangan pada awal Juli, indeks vegetasi naik cepat menuju puncak vegetatif pada awal September (sekitar dua bulan kemudian), lalu menurun hingga masa pemasakan dan panen pada pertengahan Oktober. Pada grafik tampak curah hujan 10-harian melemah di pertengahan Juli hingga awal September saat fase pertumbuhan aktif berlangsung, yang mengindikasikan musim tanam kedua di wilayah ini tetap ditopang oleh pasokan irigasi. Penurunan sesaat NDVI pada 25 September dipicu kabut tipis yang masih lolos dari penyaringan `scl`, sedangkan EVI tetap relatif stabil karena ikut memanfaatkan kanal biru untuk meredam pengaruh hamburan atmosfer [(Huete dkk., 2002)](#ref-huete-2002).

Selanjutnya, kondisi spasial NDVI di area sawah dibandingkan pada ketiga tanggal kunci tersebut.

In [ ]:
key_dates = [
    (t_flood.strftime("%Y-%m-%d"), "Penggenangan / awal tanam"),
    (t_peak.strftime("%Y-%m-%d"), "Puncak vegetatif"),
    (t_harvest.strftime("%Y-%m-%d"), "Pemasakan / panen"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
for idx, (dt_str, label) in enumerate(key_dates):
    ndvi_slice = ndvi.sel(time=dt_str).isel(time=0).where(paddy_mask)
    plot_map(
        axes[idx],
        ndvi_slice,
        f"{label}\n({dt_str})",
        cmap="RdYlGn",
        vmin=0.0,
        vmax=0.85,
        cbar_label="NDVI" if idx == 2 else None,
    )
plt.show()

Meskipun rata-rata wilayah memuncak pada 5 September, peta di atas memperlihatkan tidak seluruh petak berada pada tingkat kehijauan yang sama di tanggal tersebut.

## E. Memetakan kelompok jadwal tanam dan komposisi fase bulanan

Pada peta NDVI di bagian sebelumnya terlihat bahwa tidak seluruh petak mencapai fase yang sama pada tanggal yang sama. Untuk melihat blok mana yang bergerak lebih awal atau menyusul, setiap piksel sawah dikelompokkan berdasarkan waktu puncak NDVI masing-masing pada periode pertengahan Juli hingga pertengahan Oktober.

Batas `1 September` (`DOY 244`) dan `15 September` (`DOY 258`) digunakan sebagai pembagian praktis rentang dua mingguan di sekitar puncak utama `5 September`, sehingga kelompok blok tanam awal, utama, dan susulan terbaca seimbang.

In [ ]:
# Cari hari ke-n (Day of Year) saat tiap piksel sawah mencapai NDVI puncak
ndvi_peak_slice = ndvi.sel(time=peak_window).where(paddy_mask)
peak_idx = ndvi_peak_slice.fillna(-1).argmax(dim="time")
peak_doy_vals = pd.to_datetime(ndvi_peak_slice.time.values).dayofyear.to_numpy()
peak_doy = ndvi_peak_slice.isel(time=0).copy(data=peak_doy_vals[peak_idx.values]).where(paddy_mask)

# Kelompokkan menjadi 3 blok jadwal tanam: Awal (< 1 Sep), Utama (1-15 Sep), Akhir (> 15 Sep)
block_class = (
    paddy_mask.astype("float32")
    .where(~paddy_mask, np.where(peak_doy < 244, 1, np.where(peak_doy <= 258, 2, 3)))
    .where(paddy_mask)
)

block_meta = [
    (1, "Tanam awal", "Puncak Agustus"),
    (2, "Tanam utama", "Puncak 1–15 September"),
    (3, "Tanam akhir / susulan", "Puncak > 15 September"),
]
block_summary = pd.DataFrame(
    [
        {
            "Kelompok": nama,
            "Puncak vegetatif": ket,
            "Luas (ha)": round(float((block_class == kode).sum()) * pixel_area_ha, 1),
            "Proporsi (%)": round(100.0 * float((block_class == kode).sum()) / float(paddy_mask.sum()), 1),
        }
        for kode, nama, ket in block_meta
    ]
)
block_summary

In [ ]:
block_cmap = ListedColormap(["#f59e0b", "#16a34a", "#2563eb"])
block_legend = [
    ("#f59e0b", "Tanam awal (puncak Agu)"),
    ("#16a34a", "Tanam utama (1–15 Sep)"),
    ("#2563eb", "Tanam akhir (> 15 Sep)"),
]

fig, ax = plt.subplots(figsize=(9, 5.5))
plot_map(
    ax,
    block_class,
    "Sebaran kelompok jadwal tanam padi (musim tanam kedua 2025)",
    cmap=block_cmap,
    vmin=0.5,
    vmax=3.5,
    legend=block_legend,
)
plt.tight_layout()
plt.show()

Peta di atas memperlihatkan pergeseran waktu puncak vegetatif sekitar 2 hingga 4 minggu antarblok: sekitar separuh area sawah mengikuti jadwal utama pada awal September, hampir sepertiga memuncak lebih awal di bulan Agustus, dan seperlimanya menyusul pada akhir September hingga Oktober. Mengingat pada kurva Bagian D jarak antara puncak vegetatif (`5 September`) dan pemasakan (`12 Oktober`) berkisar 5 minggu (~37 hari), pergeseran ini berarti jendela kesiapan panen antarblok juga ikut bergeser seiring jadwal air irigasi di tiap blok.

Dinamika bulanan antarpetak tersebut juga dapat dirangkum menjadi **komposisi fase pertumbuhan per bulan** dari Juni hingga November 2025. Mei tidak disertakan karena masih membawa transisi vegetasi pascapanen musim sebelumnya sebelum penggenangan meluas pada Juni. Tiap piksel pada bulan bersangkutan dikelompokkan berdasarkan median bulanan `NDVI` dan `LSWI` terhadap bulan puncaknya:

- **Penggenangan / tanam**: sinyal air tinggi (`LSWI + 0,05 >= NDVI` atau `NDVI < 0,35`) hingga bulan puncak.
- **Vegetatif aktif**: tanaman mulai menutup (`NDVI >= 0,35` tetapi `< 0,65`) sebelum mencapai puncak.
- **Puncak vegetatif**: tutupan kanopi rapat (`NDVI >= 0,65`).
- **Pemasakan / panen**: indeks vegetasi turun di bawah `0,65` setelah melewati bulan puncak.

In [ ]:
# Hitung median bulanan NDVI & LSWI serta bulan puncak tiap piksel untuk klasifikasi fase
peak_month_vals = pd.to_datetime(ndvi_peak_slice.time.values).month.to_numpy()
peak_month = peak_month_vals[peak_idx.values]

ndvi_monthly = ndvi.where(paddy_mask).groupby("time.month").median(dim="time")
lswi_monthly = lswi.where(paddy_mask).groupby("time.month").median(dim="time")

month_names = {6: "Jun", 7: "Jul", 8: "Agu", 9: "Sep", 10: "Okt", 11: "Nov"}
phase_rows = []
for m in range(6, 12):
    n = ndvi_monthly.sel(month=m).values
    l = lswi_monthly.sel(month=m).values
    valid = paddy_mask.values & np.isfinite(n) & np.isfinite(l)
    n_tot = valid.sum()

    is_flood = valid & ((l + 0.05 >= n) | (n < 0.35)) & (m <= peak_month)
    is_peak = valid & (~is_flood) & (n >= 0.65)
    is_veg = valid & (~is_flood) & (~is_peak) & (m <= peak_month)
    is_ripen = valid & (~is_flood) & (~is_peak) & (m > peak_month)

    phase_rows.append(
        {
            "Bulan": month_names[m],
            "Penggenangan / tanam (%)": round(100.0 * is_flood.sum() / n_tot, 1),
            "Vegetatif aktif (%)": round(100.0 * is_veg.sum() / n_tot, 1),
            "Puncak vegetatif (%)": round(100.0 * is_peak.sum() / n_tot, 1),
            "Pemasakan / panen (%)": round(100.0 * is_ripen.sum() / n_tot, 1),
        }
    )

phase_df = pd.DataFrame(phase_rows)
phase_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
phase_cols = [
    ("Penggenangan / tanam (%)", "#0284c7"),
    ("Vegetatif aktif (%)", "#84cc16"),
    ("Puncak vegetatif (%)", "#15803d"),
    ("Pemasakan / panen (%)", "#d97706"),
]

bottom = np.zeros(len(phase_df))
for col, color in phase_cols:
    vals = phase_df[col].to_numpy()
    ax.bar(phase_df["Bulan"], vals, bottom=bottom, label=col.replace(" (%)", ""), color=color, width=0.58)
    bottom += vals

ax.set_title("Komposisi fase pertumbuhan area sawah per bulan (Juni–November 2025)", pad=26)
ax.set_ylabel("Proporsi luas sawah (%)")
ax.set_ylim(0, 105)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=4, frameon=False, fontsize=9)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Grafik di atas merangkum pergeseran fase musim tanam kedua secara utuh: penggenangan mendominasi Juni hingga Juli, puncak vegetatif menguat sejak Agustus hingga lebih dari 80% area pada September, lalu pemasakan dan panen mendominasi Oktober hingga November. Ringkasan per bulan dan pembagian blok seperti ini berguna sebagai alat bantu pembanding di lapangan untuk mengidentifikasi kelompok petak yang panen lebih dulu atau masih membutuhkan pasokan air, serta menentukan jadwal dan lokasi survei lapangan agar sesuai kondisi aktual tiap blok.

Seluruh kelompok jadwal tanam digabungkan ke dalam satu peta interaktif di atas citra satelit agar lokasi tiap blok dapat diperiksa langsung hingga tingkat petak.

In [ ]:
m = aoi.explore(
    tiles=basemaps.Esri.WorldImagery,
    name="Batas AoI",
    style_kwds={"fill": False, "color": "white", "weight": 2},
)

for kode, nama, color in [
    (1, "Tanam awal (puncak Agu)", "#f59e0b"),
    (2, "Tanam utama (1–15 Sep)", "#16a34a"),
    (3, "Tanam akhir (> 15 Sep)", "#2563eb"),
]:
    layer = (block_class == kode).where(block_class == kode).astype("float32")
    layer.odc.explore(map=m, name=nama, cmap=ListedColormap([color]), vmin=0, vmax=1, opacity=0.8)

folium.LayerControl(collapsed=False).add_to(m)
m

## F. Pengembangan analisis untuk pemantauan padi

Analisis di atas dapat dilanjutkan untuk kebutuhan pemantauan operasional melalui beberapa langkah berikut:

- **Memadukan hasil deteksi dengan poligon resmi (Lahan Baku Sawah atau batas daerah irigasi).** Skrip saat ini menentukan area sawah hanya dari kriteria spektral (`paddy_mask`). Jika digabungkan dengan poligon Lahan Baku Sawah atau batas petak tersier irigasi, rekap fase bulanan dan kelompok jadwal tanam dapat langsung dihitung per desa, kecamatan, atau golongan air.
- **Melanjutkan ke musim hujan menggunakan citra radar (`s1_rtc`).** Analisis di atas mengambil musim tanam kedua pada periode kemarau karena citra optis `s2_l2a` masih cukup sering bersih dari awan (`24` dari `46` akuisisi). Untuk memantau musim tanam pertama pada periode hujan yang berawan tebal, metode dapat diperluas menggunakan citra SAR Sentinel-1 di Open Data Cube yang tetap mampu menangkap sinyal penggenangan dan pertumbuhan kanopi di balik awan.
- **Memetakan intensitas tanam tahunan dan pergeseran antartahun.** Menjalankan pengelompokan fase di atas sepanjang satu tahun penuh memungkinkan identifikasi petak yang ditanam sekali dibanding dua kali setahun, serta membandingkan pergeseran jadwal blok tanam dan curah hujan antara tahun normal dan tahun kering (misalnya El Niño 2023).
- **Membangun rujukan multitahun untuk mendeteksi anomali pertumbuhan.** Arsip seri waktu `NDVI`, `EVI`, dan `LSWI` pada blok yang sama selama beberapa tahun dapat dijadikan pembanding normal, sehingga penurunan indeks yang mencolok di fase tertentu dapat ditandai lebih awal untuk dicek penyebabnya di lapangan (seperti kekurangan air, kebanjiran, atau serangan organisme pengganggu tumbuhan).
- **Membangun model penduga hasil panen dengan data sampel lapangan (*ubinan*).** Nilai puncak kehijauan (`EVI` maksimum) dan total akumulasi indeks sepanjang masa tumbuh tiap petak dari seri waktu di atas dapat dijadikan variabel prediktor, lalu dilatih terhadap data timbangan hasil panen nyata (`ton/ha`) dari survei *ubinan* BPS atau Kementerian Pertanian. Model yang sudah terkalibrasi pada musim dan varietas setempat tersebut (sering ditambah indeks *red-edge* `NDRE` karena `NDVI` cepat jenuh pada kanopi rapat) kemudian dapat dipakai untuk memperkirakan produktivitas panen di petak sawah lainnya.

## Referensi

(ref-rouse-1974)=
1. Rouse, J. W., Jr., Haas, R. H., Schell, J. A., & Deering, D. W. (1974). Monitoring vegetation systems in the Great Plains with ERTS. *Third Earth Resources Technology Satellite-1 Symposium*, Volume 1, Section A, NASA Goddard Space Flight Center, 309–317.

(ref-huete-2002)=
2. Huete, A., Didan, K., Miura, T., Rodriguez, E. P., Gao, X., & Ferreira, L. G. (2002). Overview of the radiometric and biophysical performance of the MODIS vegetation indices. *Remote Sensing of Environment*, 83(1–2), 195–213. https://doi.org/10.1016/S0034-4257(02)00096-2

(ref-xiao-2005)=
3. Xiao, X., Boles, S., Liu, J., Zhuang, D., Frolking, S., Li, C., Salas, W., & Moore III, B. (2005). Mapping paddy rice agriculture in southern China using multi-temporal MODIS images. *Remote Sensing of Environment*, 95(4), 480–492. https://doi.org/10.1016/j.rse.2004.12.009

(ref-xiao-2006)=
4. Xiao, X., Boles, S., Frolking, S., Li, C., Babu, J. Y., Salas, W., & Moore, B. (2006). Mapping paddy rice agriculture in South and Southeast Asia using multi-temporal MODIS images. *Remote Sensing of Environment*, 100(1), 95–113. https://doi.org/10.1016/j.rse.2005.10.004

(ref-hersbach-2020)=
5. Hersbach, H., Bell, B., Berrisford, P., Hirahara, S., Horányi, A., Muñoz-Sabater, J., Nicolas, J., Peubey, C., Radu, R., Schepers, D., Simmons, A., Soci, C., Abdalla, S., Abellan, X., Balsamo, G., Bechtold, P., Biavati, G., Bidlot, J., Bonavita, M., ... Thépaut, J.-N. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, 146(730), 1999–2049. https://doi.org/10.1002/qj.3803